# Real-Time Vision — Edge Deployment

## Basic Concept

A training-time vision model is a floating-point monster. 100M parameters, 10 GFLOPs per forward pass, 2 GB of VRAM. None of that fits on a phone, a car's infotainment unit, an industrial camera, or a drone. **Shipping a vision system means fitting the same predictions into a budget that is 100x smaller.**

Three knobs do most of the work: model choice (a smaller architecture with the same recipe), quantisation (INT8 instead of FP32), and the inference runtime (ONNX Runtime, TensorRT, Core ML, TFLite).

### The three budgets

```mermaid
flowchart LR
    M["Model"] --> LAT["Latency<br/>ms per image"]
    M --> MEM["Memory<br/>peak MB"]
    M --> PWR["Power<br/>mJ per inference"]

    LAT --> SHIP["Ship / no-ship<br/>decision"]
    MEM --> SHIP
    PWR --> SHIP

    style LAT fill:#fecaca,stroke:#dc2626
    style MEM fill:#fef3c7,stroke:#d97706
    style PWR fill:#dbeafe,stroke:#2563eb
```

- **Latency**: p50, p95, p99. Averaging only p50 hides tail behaviour that matters for real-time systems.
- **Peak memory**: **the maximum the device ever sees**, not the steady-state average. Matters because OOMs are fatal on embedded targets.
- **Power / energy**: millijoules per inference on a battery-powered device. Often proxied by CPU/GPU utilisation * time.

A table of (model, latency, memory, accuracy) is what an edge decision is made from. Every cell is measured on the target device, not the workstation.

### Measurement discipline

Three rules that every edge profile should follow:

1. **Warm up** the model with 5-10 dummy forward passes before measuring. Cold caches and JIT compilation produce unrepresentative first numbers.
2. **Synchronise** GPU workloads with `torch.cuda.synchronize()` before and after the timed block. Without this you measure kernel dispatch, not kernel execution.
3. **Fix input sizes** to the production resolution. Latency on 224x224 is not latency on 512x512.

### FLOPs as a proxy

FLOPs (floating-point operations per inference) is a cheap, device-independent proxy for latency. Useful for architecture comparison, misleading as absolute wall-clock. A model with 10% more FLOPs can be 2x faster in practice because it uses hardware-friendly ops (depthwise convs compile well, large 7x7 convs do not).

Rule: use FLOPs for architecture search, use on-device latency for deployment decisions.

### Quantisation in one paragraph

Replace FP32 weights and activations with INT8. Model size drops 4x, memory bandwidth drops 4x, compute drops 2-4x on hardware that has INT8 kernels (every modern mobile SoC, every NVIDIA GPU with Tensor Cores). Accuracy loss on vision tasks is **typically 0.1-1 percentage points with post-training static quantisation.**

### Pruning and distillation

- **Pruning** — remove unimportant weights (magnitude-based) or channels (structured). Works well on overparameterised models; less useful on already-compact architectures.
- **Distillation** — train a small student to mimic a large teacher's logits. Often recovers most of the accuracy lost by shrinking the model. Standard for production edge models.


# Build your Own

## Latency Measurment

In [ ]:
import time
import torch

def measure_latency(model, input_shape, device="mps", warmup=10, iters=50):
    model = model.to(device).eval()
    x = torch.randn(input_shape).to(device)

    with torch.no_grad():
        for _ in range(warmup):
            model(x)
        if device == "cuda":
            torch.cuda.synchronize()
        elif device == "mps":
            torch.mps.synchronize()

        times = []
        for _ in range(iters):
            if device == "cuda":
                torch.cuda.synchronize()
            elif device == "mps":
                torch.mps.synchronize()

            start_time = time.perf_counter()
            model(x)

            if device == "cuda":
                torch.cuda.synchronize()
            elif device == "mps":
                torch.mps.synchronize()

            end_time = time.perf_counter()
            times.append((end_time - start_time) * 1000)
    
    times.sort()

    return {
        "p50_ms": times[len(times) // 2],
        "p95_ms": times[int(len(times) * 0.95)],
        "p99_ms": times[int(len(times) * 0.99)],
        "mean_ms": sum(times) / len(times),
    }


## Parameter and FLOP counts

In [ ]:
def paramter_count(model):
    return sum(p.numel() for p in model.parameters())

def flops_estimate(model, input_shape):
    """
    Rough FLOP count for a conv/linear-only model. For production use `fvcore` or `ptflops`.
    """
    total = 0
    def conv_hook(m, inp, out):
        nonlocal total
        c_out, c_in, kh, kw = m.weight.shape
        h, w = out.shape[-2:]
        total += 2 * c_in * c_out * kh * kw * h * w
    
    def linear_hook(m, inp, out):
        nonlocal total
        total += 2 * m.in_features * m.out_features

    hooks = []
    for m in model.modules():
        if isinstance(m, torch.nn.Conv2d):
            hooks.append(m.register_forward_hook(conv_hook))
        elif isinstance(m, torch.nn.Linear):
            hooks.append(m.register_forward_hook(linear_hook))
    
    model.eval()

    with torch.no_grad():
        model(torch.randn(input_shape))

    for h in hooks:
        h.remove()

    return total

# Post-training quantisation (torchao)

In [1]:
def quantise_ptq(model, calibration_loader=None):
    """
    Post-training quantisation via torchao.quantize_ (replaces deprecated
    torch.ao.quantization.prepare / convert).

    Applies int8 dynamic activations + int8 weights to Linear layers.
    calibration_loader is optional for this config (scales estimated at
    runtime); if given, we still run a forward pass for a familiar PTQ flow.
    """
    from torchao.quantization import Int8DynamicActivationInt8WeightConfig, quantize_

    model = model.eval().cpu()
    if calibration_loader is not None:
        with torch.no_grad():
            for batch in calibration_loader:
                x = batch[0] if isinstance(batch, (tuple, list)) else batch
                model(x.cpu())
    quantize_(model, Int8DynamicActivationInt8WeightConfig())
    return model


# smoke test: no DeprecationWarning from torch.ao.quantization.prepare
import warnings
import torch.nn as nn

_m = nn.Sequential(nn.Flatten(), nn.Linear(3 * 8 * 8, 16), nn.ReLU(), nn.Linear(16, 5)).eval()
_loader = [(torch.randn(4, 3, 8, 8),)]
with warnings.catch_warnings(record=True) as _w:
    warnings.simplefilter("always")
    _q = quantise_ptq(_m, _loader)
    _out = _q(torch.randn(2, 3, 8, 8))
assert _out.shape == (2, 5)
assert not any(issubclass(x.category, DeprecationWarning) for x in _w), _w
print("quantise_ptq ok", _out.shape)

    with torch.no_grad():
        for x, _ in calibration_loader:
            model(x)
    tq.convert(model, inplace=True)
    return model

IndentationError: unexpected indent (830882673.py, line 36)